In [0]:
from pytickersymbols import PyTickerSymbols
import yfinance as yf
from pyspark.sql.types import StructType, StructField, DateType, DoubleType, LongType, StringType, BooleanType
import pandas as pd

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS yfinance_pipeline_prod;
USE CATALOG yfinance_pipeline_prod;
CREATE SCHEMA IF NOT EXISTS stocks_dataset;
USE SCHEMA stocks_dataset;

In [0]:
stock_data = PyTickerSymbols()

dowjones_stocks = stock_data.get_stocks_by_index('DOW JONES')
dowjones_symbols = [stock['symbol'] for stock in dowjones_stocks]
nasdaq100_stocks = stock_data.get_stocks_by_index('NASDAQ 100')
nasdaq100_symbols = [stock['symbol'] for stock in nasdaq100_stocks]
sp500_stocks = stock_data.get_stocks_by_index('S&P 500')
sp500_symbols = [stock['symbol'] for stock in sp500_stocks]

In [0]:
# Collect unique symbols from all major stock lists
collected_symbols = []

# for symbol_list in [nasdaq100_symbols, dowjones_symbols, sp500_symbols, sp600_symbols]:
for symbol_list in [nasdaq100_symbols, dowjones_symbols, sp500_symbols]:
    for symbol in symbol_list:
        if symbol not in collected_symbols and "." not in symbol:
            collected_symbols.append(symbol)

print(f"Total unique symbols: {len(collected_symbols)}")
print(collected_symbols[:20])  # Show first 20 for preview

In [0]:
# sample_tickers = yf.Tickers("AAPL MSFT GOOGL GOOG AMZN META TSLA NVDA")

In [0]:
def load_stocks_price(
    collected_symbols,
    table_name,
    fetch_period,
    batch_size=100
):
    schema = StructType([
        StructField("symbol", StringType(), True),
        StructField("date", DateType(), True),
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", LongType(), True)
    ])

    for i in range(0, len(collected_symbols), batch_size):
        # Get the current slice of 100 tickers
        current_batch = collected_symbols[i:i + batch_size]
        print(f"Processing batch {i//batch_size + 1}: {current_batch[0]} to {current_batch[-1]}...")

        # Initialize the Tickers object for this batch
        tickers = yf.Tickers(current_batch)
        dfs = []
        # Example: Accessing data for each ticker in the current batch
        for symbol in current_batch:
            try:
                # Note: .info is slow. If you only need prices, use .history instead.
                # info_data = tickers_obj.tickers[symbol].info

                hist = tickers.tickers[symbol].history(period="max").reset_index()
                hist['symbol'] = symbol
                hist = hist[['symbol', 'Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
                hist.columns = ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']
                hist['date'] = hist['date'].dt.date
                dfs.append(hist)

            except Exception as e:
                print(f"Error fetching {symbol}: {e}")

        stocks_hist = pd.concat(dfs, ignore_index=True)

        stocks_hist_spark = spark.createDataFrame(stocks_hist, schema=schema)

        # Create or replace table stocks_price_history
        stocks_hist_spark.write.mode("append").saveAsTable(table_name)

    return

In [0]:
if not spark.catalog.tableExists("stocks_price_history_raw"):
    load_stocks_price(
        collected_symbols=collected_symbols,
        table_name="stocks_price_history_raw",
        fetch_period="max"
    )
    print("complete")
if not spark.catalog.tableExists("indexes_price_history"):
    load_stocks_price(
        collected_symbols=["^GSPC", "^DJI", "^NDX"],
        table_name="indexes_price_history",
        fetch_period="max"
    )
    print("complete")
